In [1]:
# COMMAND: Install required packages

!pip install -q pandas numpy scikit-learn joblib

print("Required packages installed successfully!")

Required packages installed successfully!


In [3]:
# COMMAND: Mount Google Drive

from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully!


In [4]:
# COMMAND: Import required libraries

import pandas as pd
import numpy as np
import os
import joblib

print("All libraries imported successfully!")

All libraries imported successfully!


In [5]:
# COMMAND: Define project and model paths

project_path = (
    "/content/drive/MyDrive/"
    "Alumni_Donor_Propensity_Forecaster"
)

model_path = os.path.join(
    project_path,
    "models",
    "alumni_donor_model_pipeline.pkl"
)

print("Project Path:")
print(project_path)

print("\nModel Path:")
print(model_path)

Project Path:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster

Model Path:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/models/alumni_donor_model_pipeline.pkl


In [6]:
# COMMAND: Check whether trained model exists

import os

if os.path.exists(model_path):

    print("SUCCESS!")
    print("Trained model found.")
    print(model_path)

else:

    print("ERROR!")
    print("Trained model was not found.")
    print(model_path)

SUCCESS!
Trained model found.
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/models/alumni_donor_model_pipeline.pkl


In [7]:
# COMMAND: Load the trained ML pipeline

import joblib

model = joblib.load(
    model_path
)

print("Trained model loaded successfully!")

print("\nModel type:")
print(type(model))

Trained model loaded successfully!

Model type:
<class 'sklearn.pipeline.Pipeline'>


In [8]:
# COMMAND: Create new alumni input data

new_alumni = pd.DataFrame([{

    "Age": 45,

    "Gender": "Male",

    "Graduation_Year": 2005,

    "Degree_Level": "Master",

    "Major": "Business",

    "Alumni_Status": "Active",

    "Email_Available": "Yes",

    "Phone_Available": "Yes",

    "Wealth_Rating": "A",

    "Income_Bracket": "$100K-$150K",

    "Event_Attendance": 8,

    "Email_Open_Rate": 75.5,

    "Location": "New York, USA",

    "Consecutive_Giving_Years": 5,

    "Donation_Last_Year_2024": 1500.00,

    "Total_Lifetime_Giving": 10000.00

}])

print("New alumni record created!")

display(new_alumni)

New alumni record created!


,Age,Gender,Graduation_Year,Degree_Level,Major,Alumni_Status,Email_Available,Phone_Available,Wealth_Rating,Income_Bracket,Event_Attendance,Email_Open_Rate,Location,Consecutive_Giving_Years,Donation_Last_Year_2024,Total_Lifetime_Giving
0,45,Male,2005,Master,Business,Active,Yes,Yes,A,$100K-$150K,8,75.5,"New York, USA",5,1500.0,10000.0


In [9]:
# COMMAND: Validate input columns

expected_columns = [
    "Age",
    "Gender",
    "Graduation_Year",
    "Degree_Level",
    "Major",
    "Alumni_Status",
    "Email_Available",
    "Phone_Available",
    "Wealth_Rating",
    "Income_Bracket",
    "Event_Attendance",
    "Email_Open_Rate",
    "Location",
    "Consecutive_Giving_Years",
    "Donation_Last_Year_2024",
    "Total_Lifetime_Giving"
]

missing_columns = [
    column
    for column in expected_columns
    if column not in new_alumni.columns
]

extra_columns = [
    column
    for column in new_alumni.columns
    if column not in expected_columns
]

print("Missing columns:", missing_columns)

print("Extra columns:", extra_columns)

if not missing_columns and not extra_columns:

    print("\nInput validation successful!")

else:

    print("\nWARNING: Input columns need to be checked!")

Missing columns: []
Extra columns: []

Input validation successful!


In [10]:
# COMMAND: Generate Yes/No donor prediction

prediction = model.predict(
    new_alumni
)[0]

print("Raw Model Prediction:")
print(prediction)

Raw Model Prediction:
1


In [11]:
# COMMAND: Convert numerical prediction into Yes/No

prediction_label = (
    "Yes"
    if prediction == 1
    else "No"
)

print("Donation Prediction:")
print(prediction_label)

Donation Prediction:
Yes


In [12]:
# COMMAND: Calculate probability of donating

probability = model.predict_proba(
    new_alumni
)[0][1]

print("Donation Probability:")
print(probability)

Donation Probability:
0.86


In [13]:
# COMMAND: Convert probability into propensity score

propensity_score = probability * 100

print(
    f"Propensity Score: {propensity_score:.2f}%"
)

Propensity Score: 86.00%


In [14]:
# COMMAND: Categorize donation propensity

if propensity_score >= 70:

    category = "High"

elif propensity_score >= 40:

    category = "Medium"

else:

    category = "Low"

print("Propensity Category:")
print(category)

Propensity Category:
High


In [15]:
# COMMAND: Generate human-readable interpretation

interpretation_map = {

    "High":
        "Likely to Donate",

    "Medium":
        "Moderate Donation Propensity",

    "Low":
        "Less Likely to Donate"
}

interpretation = interpretation_map[
    category
]

print("Interpretation:")
print(interpretation)

Interpretation:
Likely to Donate


In [16]:
# COMMAND: Create final prediction response

result = {

    "prediction":
        prediction_label,

    "donation_probability":
        round(probability, 4),

    "propensity_score":
        round(propensity_score, 2),

    "category":
        category,

    "interpretation":
        interpretation
}

print("FINAL PREDICTION")
print("=" * 50)

for key, value in result.items():

    print(
        f"{key}: {value}"
    )

FINAL PREDICTION
prediction: Yes
donation_probability: 0.86
propensity_score: 86.0
category: High
interpretation: Likely to Donate


In [17]:
# COMMAND: Display prediction in a table

result_table = pd.DataFrame({

    "Prediction": [
        prediction_label
    ],

    "Donation Probability (%)": [
        round(probability * 100, 2)
    ],

    "Propensity Score": [
        round(propensity_score, 2)
    ],

    "Category": [
        category
    ],

    "Interpretation": [
        interpretation
    ]
})

display(result_table)

,Prediction,Donation Probability (%),Propensity Score,Category,Interpretation
0,Yes,86.0,86.0,High,Likely to Donate


In [18]:
# COMMAND: Save prediction result to Google Drive

results_path = os.path.join(
    project_path,
    "results",
    "alumni_prediction_result.csv"
)

result_table.to_csv(
    results_path,
    index=False
)

print("Prediction result saved successfully!")

print("\nFile:")
print(results_path)

Prediction result saved successfully!

File:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/results/alumni_prediction_result.csv


In [19]:
# COMMAND: Verify prediction result file

import os

if os.path.exists(results_path):

    print("Prediction result file exists!")

    print("\nLocation:")
    print(results_path)

else:

    print("Prediction result file was not created.")

Prediction result file exists!

Location:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/results/alumni_prediction_result.csv


In [20]:
# COMMAND: Final inference verification

print("=" * 70)
print("       ALUMNI DONOR PROPENSITY FORECASTER")
print("             INFERENCE COMPLETED")
print("=" * 70)

print("\nAlumni Prediction:")
print(
    f"Prediction           : {prediction_label}"
)

print(
    f"Donation Probability : {probability * 100:.2f}%"
)

print(
    f"Propensity Score     : {propensity_score:.2f}"
)

print(
    f"Category             : {category}"
)

print(
    f"Interpretation       : {interpretation}"
)

print("\n" + "=" * 70)
print("PRETRAINED MODEL TEST SUCCESSFUL!")
print("=" * 70)

       ALUMNI DONOR PROPENSITY FORECASTER
             INFERENCE COMPLETED

Alumni Prediction:
Prediction           : Yes
Donation Probability : 86.00%
Propensity Score     : 86.00
Category             : High
Interpretation       : Likely to Donate

PRETRAINED MODEL TEST SUCCESSFUL!
